# Trabajo Práctico N.º 4 — Taller de Programación
## Clasificando informales en la EPH: regularización y CART

**Grupo:** 1  
**Estudiante:** Estefanía Embarbe  
**Períodos:** 4.º trimestre de 2024 y 4.º trimestre de 2025

### Objetivo
Predecir la condición de informalidad laboral en 2025 para trabajadores asalariados observados longitudinalmente en 2024 y 2025. La variable objetivo toma valor 1 cuando el asalariado no registra descuento jubilatorio (`PP07H = 2`) y 0 cuando sí lo registra (`PP07H = 1`).

La especificación principal utiliza características socioeconómicas y laborales observadas en 2025 junto con el rezago de informalidad de 2024. Se comparan Logit sin penalización, LASSO (L1), Ridge (L2) y CART. La evaluación se realiza con 5-fold cross-validation y métricas de clasificación.

### Criterio de especificación
Se prioriza una matriz económicamente informativa: capital humano, intensidad e inserción laboral, características del establecimiento, estabilidad, rama productiva, ocupación, calificación, ingreso y territorio. Se excluyen variables que revelan directamente la condición objetivo o presentan solapamiento conceptual fuerte con la protección social.


## 0. Consigna cubierta por esta notebook

- **A.1** Trayectorias de coeficientes para LASSO y Ridge en la grilla $\lambda=10^n$, $n\in\{-5,\ldots,5\}$.
- **A.2** Selección de $\lambda_{CV}$ mediante 5-fold cross-validation y boxplots del error de clasificación.
- **A.3** Tabla comparativa de coeficientes: Logit, LASSO y Ridge.
- **B.1** Selección de `ccp_alpha` de CART mediante 5-fold cross-validation.
- **B.2** Árbol podado, importancia de predictores y comparación con la selección de LASSO.
- **C.1** Matrices de confusión, ROC y métricas de desempeño para los cuatro modelos.
- **C.2** Discusión de focalización para política pública y evaluación de informales en una región argentina.
- **D** Reflexión sobre IA, registro de prompts y tiempo de trabajo.


## 1. Configuración y carga de datos

Las bases individuales de EPH deben estar cargadas en `/content/` con los nombres indicados debajo. Todas las salidas relevantes se guardan en `/content/TP4_outputs/`.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

RANDOM_STATE = 42
OUTPUT_DIR = Path("/content/TP4_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
ruta_2024 = Path("/content/usu_individual_T424.xlsx")
ruta_2025 = Path("/content/usu_individual_T425.xlsx")

assert ruta_2024.exists(), "No se encontró usu_individual_T424.xlsx"
assert ruta_2025.exists(), "No se encontró usu_individual_T425.xlsx"

df_2024 = pd.read_excel(ruta_2024)
df_2025 = pd.read_excel(ruta_2025)

df_2024.columns = df_2024.columns.str.strip().str.upper()
df_2025.columns = df_2025.columns.str.strip().str.upper()

print("EPH 2024T4:", df_2024.shape)
print("EPH 2025T4:", df_2025.shape)


## 2. Universo de análisis y variable objetivo

Se restringe la muestra a ocupados asalariados (`ESTADO = 1`, `CAT_OCUP = 3`) con respuesta válida en `PP07H`. Se define `informal = 1` cuando no hay descuento jubilatorio y `informal = 0` cuando sí lo hay.


In [ ]:
def preparar_universo(df, periodo):
    datos = df.copy()
    datos = datos[datos["ESTADO"] == 1].copy()
    datos = datos[datos["CAT_OCUP"] == 3].copy()
    datos = datos[datos["PP07H"].isin([1, 2])].copy()
    datos["informal"] = (datos["PP07H"] == 2).astype(int)
    datos["periodo"] = periodo
    return datos

asal_2024 = preparar_universo(df_2024, "2024T4")
asal_2025 = preparar_universo(df_2025, "2025T4")

print("Asalariados clasificables 2024:", len(asal_2024))
print("Asalariados clasificables 2025:", len(asal_2025))
print("\nInformalidad 2024 (%):")
print(asal_2024["informal"].value_counts(normalize=True).sort_index().mul(100).round(2))
print("\nInformalidad 2025 (%):")
print(asal_2025["informal"].value_counts(normalize=True).sort_index().mul(100).round(2))


## 3. Construcción de predictores

La matriz se construye con variables que tienen una interpretación directa para la informalidad laboral. Se conserva `educ` en años de educación —creada en el TP2— y no se agregan simultáneamente dummies de nivel educativo para evitar redundancia.

### Bloques
- **Capital humano y hogar:** edad, edad², años de educación, tamaño del hogar.
- **Inserción e intensidad laboral:** horas, pluriempleo, deseo/búsqueda de más horas, búsqueda de otro empleo, intensidad.
- **Puesto y establecimiento:** tipo y tamaño del establecimiento.
- **Estabilidad:** antigüedad y duración del empleo.
- **Estructura productiva:** rama CAES, grupo ocupacional CNO y calificación.
- **Posición socioeconómica:** decil de ingreso laboral.
- **Demografía y territorio:** sexo, migración, región y tamaño del aglomerado.


In [ ]:
def construir_educ(df):
    ch12 = pd.to_numeric(df["CH12"], errors="coerce")
    ch13 = pd.to_numeric(df["CH13"], errors="coerce")
    ch14 = pd.to_numeric(df["CH14"], errors="coerce")

    educ = pd.Series(np.nan, index=df.index, dtype="float64")

    if "CH10" in df.columns:
        educ.loc[df["CH10"] == 3] = 0

    anios_nivel_completo = {
        1: 0, 2: 7, 3: 9, 4: 12,
        5: 12, 6: 15, 7: 17, 8: 19,
    }
    for nivel, anios in anios_nivel_completo.items():
        educ.loc[(ch12 == nivel) & (ch13 == 1)] = anios

    base_nivel = {
        1: 0, 2: 0, 3: 0, 4: 7,
        5: 9, 6: 12, 7: 12, 8: 17,
    }
    ultimo_anio_valido = ch14.between(0, 9)
    for nivel, base in base_nivel.items():
        mask = (ch12 == nivel) & (ch13 == 2) & ultimo_anio_valido
        educ.loc[mask] = base + ch14.loc[mask]

    return educ


def agregar_nhogar(df_completa, df_asal):
    tam_hogar = (
        df_completa
        .groupby(["CODUSU", "NRO_HOGAR"])["COMPONENTE"]
        .nunique()
        .rename("nhogar")
        .reset_index()
    )
    return df_asal.merge(
        tam_hogar,
        on=["CODUSU", "NRO_HOGAR"],
        how="left",
        validate="many_to_one",
    )


def agrupar_tamano_establecimiento(x):
    if pd.isna(x) or x == 99:
        return "sin_info"
    if x == 0:
        return "no_aplica"
    if x == 1:
        return "1_persona"
    if 2 <= x <= 5:
        return "2a5"
    if x == 6:
        return "6a10"
    if x in [7, 8]:
        return "11a40"
    if x == 9:
        return "41a100"
    if x in [10, 11, 12]:
        return "101omas"
    return "sin_info"


def agrupar_rama_caes(codigo):
    if pd.isna(codigo):
        return "sin_info"
    try:
        codigo = str(int(codigo)).zfill(4)
        division = int(codigo[:2])
    except Exception:
        return "sin_info"

    if division in [1, 2, 3]: return "agro_pesca"
    if 5 <= division <= 9: return "mineria"
    if 10 <= division <= 33: return "industria"
    if division == 35: return "energia"
    if 36 <= division <= 39: return "agua_saneamiento"
    if division == 40: return "construccion"
    if division in [45, 48]: return "comercio"
    if 49 <= division <= 53: return "transporte"
    if 55 <= division <= 56: return "alojamiento_comidas"
    if 58 <= division <= 63: return "informacion_comunicaciones"
    if 64 <= division <= 66: return "finanzas"
    if division == 68: return "inmobiliarias"
    if 69 <= division <= 75: return "profesionales_tecnicas"
    if 77 <= division <= 82: return "administrativas_apoyo"
    if division in [83, 84]: return "administracion_publica"
    if division == 85: return "educacion"
    if 86 <= division <= 88: return "salud_servicios_sociales"
    if 90 <= division <= 93: return "arte_recreacion"
    if 94 <= division <= 96: return "otros_servicios"
    if division in [97, 98]: return "servicio_domestico"
    if division == 99: return "organismos_extraterritoriales"
    return "sin_info"


def preparar_codigo_cno(x):
    if pd.isna(x):
        return None
    try:
        return str(int(x)).zfill(5)
    except Exception:
        return None


In [ ]:
def construir_predictores(df_completa, df_asal):
    datos = agregar_nhogar(df_completa, df_asal.copy())

    # Capital humano y demografía
    datos["edad"] = pd.to_numeric(datos["CH06"], errors="coerce")
    datos.loc[datos["edad"] < 0, "edad"] = np.nan
    datos["edad2"] = datos["edad"] ** 2
    datos["educ"] = construir_educ(datos)

    datos["mujer"] = np.where(
        datos["CH04"] == 2, 1,
        np.where(datos["CH04"] == 1, 0, np.nan)
    )

    datos["migrante"] = np.select(
        [datos["CH15"].isin([1, 2]), datos["CH15"].isin([3, 4, 5])],
        [0, 1],
        default=np.nan,
    )

    # Intensidad e inserción laboral
    h_principal = pd.to_numeric(datos["PP3E_TOT"], errors="coerce").replace(999, np.nan)
    h_otras = pd.to_numeric(datos["PP3F_TOT"], errors="coerce").replace(999, np.nan).fillna(0)
    datos["horas_totales"] = h_principal + h_otras
    datos.loc[~datos["horas_totales"].between(0, 168), "horas_totales"] = np.nan

    datos["pluriempleo"] = np.where(
        datos["PP03C"] == 2, 1,
        np.where(datos["PP03C"] == 1, 0, np.nan)
    )
    datos["quiere_mas_horas"] = np.where(
        datos["PP03G"] == 1, 1,
        np.where(datos["PP03G"] == 2, 0, np.nan)
    )
    datos["busco_mas_horas"] = np.where(
        datos["PP03I"] == 1, 1,
        np.where(datos["PP03I"] == 2, 0, np.nan)
    )
    datos["busco_otro_empleo"] = np.where(
        datos["PP03J"] == 1, 1,
        np.where(datos["PP03J"] == 2, 0, np.nan)
    )
    datos["intensidad"] = datos["INTENSI"].map({
        1: "subocupado",
        2: "pleno",
        3: "sobreocupado",
        4: "no_trabajo_semana",
    })

    # Puesto y establecimiento
    datos["tipo_establecimiento"] = datos["PP04A"].map({
        1: "estatal", 2: "privado", 3: "otro"
    })
    datos["tam_establecimiento"] = datos["PP04C"].apply(agrupar_tamano_establecimiento)

    # Estabilidad laboral
    datos["antiguedad"] = datos["PP07A"].map({
        0: "no_aplica",
        1: "menos_1_mes",
        2: "1a3_meses",
        3: "3a6_meses",
        4: "6a12_meses",
        5: "1a5_anios",
        6: "mas_5_anios",
        9: "sin_info",
    })
    datos["tipo_duracion_empleo"] = datos["PP07C"].map({
        0: "no_aplica",
        1: "temporal",
        2: "permanente",
        9: "sin_info",
    })

    # Ingreso
    datos["decil_ingreso"] = pd.to_numeric(datos["DECOCUR"], errors="coerce")
    datos.loc[datos["decil_ingreso"] == 12, "decil_ingreso"] = np.nan

    # Territorio
    datos["region"] = datos["REGION"].map({
        1: "gba", 40: "noa", 41: "nea", 42: "cuyo", 43: "pampeana", 44: "patagonia"
    })
    datos["aglomerado_grande"] = np.where(
        datos["MAS_500"] == "S", 1,
        np.where(datos["MAS_500"] == "N", 0, np.nan)
    )

    # Rama productiva
    datos["rama_sector"] = datos["PP04B_COD"].apply(agrupar_rama_caes)

    # Ocupación y calificación
    codigo_cno = datos["PP04D_COD"].apply(preparar_codigo_cno)
    mapa_caracter = {
        "0": "direccion",
        "1": "gestion_administrativa_legal",
        "2": "gestion_financiera_contable",
        "3": "comercio_transporte_comunicaciones",
        "4": "servicios_sociales_basicos",
        "5": "servicios_varios",
        "6": "produccion_agropecuaria",
        "7": "extractiva_energia_construccion",
        "8": "industria_reparacion",
        "9": "instalacion_mantenimiento_tecnologia",
    }
    mapa_calificacion = {
        "1": "profesional",
        "2": "tecnica",
        "3": "operativa",
        "4": "no_calificada",
    }

    datos["grupo_ocupacional"] = codigo_cno.str[0].map(mapa_caracter).fillna("sin_info")
    datos["calificacion_ocupacional"] = codigo_cno.str[-1].map(mapa_calificacion).fillna("sin_info")

    return datos

base_X_2024 = construir_predictores(df_2024, asal_2024)
base_X_2025 = construir_predictores(df_2025, asal_2025)

print("Base 2024:", base_X_2024.shape)
print("Base 2025:", base_X_2025.shape)


In [ ]:
features_numericas = [
    "edad",
    "edad2",
    "educ",
    "horas_totales",
    "nhogar",
    "decil_ingreso",
]

features_binarias = [
    "mujer",
    "migrante",
    "pluriempleo",
    "quiere_mas_horas",
    "busco_mas_horas",
    "busco_otro_empleo",
    "aglomerado_grande",
]

features_categoricas = [
    "intensidad",
    "tipo_establecimiento",
    "tam_establecimiento",
    "antiguedad",
    "tipo_duracion_empleo",
    "region",
    "rama_sector",
    "grupo_ocupacional",
    "calificacion_ocupacional",
]

features_economicas = features_numericas + features_binarias + features_categoricas

print("Predictores contemporáneos conceptuales:", len(features_economicas))
for i, var in enumerate(features_economicas, 1):
    print(f"{i:02d}. {var}")

faltan_2024 = [v for v in features_economicas if v not in base_X_2024.columns]
faltan_2025 = [v for v in features_economicas if v not in base_X_2025.columns]

assert not faltan_2024, f"Faltan variables en 2024: {faltan_2024}"
assert not faltan_2025, f"Faltan variables en 2025: {faltan_2025}"
print("\nOK: misma definición de predictores en 2024 y 2025.")


## 4. Base longitudinal definitiva

Se vincula a cada trabajador observado en 2025 con su condición de informalidad en 2024 utilizando `CODUSU`, `NRO_HOGAR` y `COMPONENTE`. La matriz principal utiliza los predictores contemporáneos de 2025 y agrega `informal_2024` como historia laboral.

No se eliminan observaciones por faltantes de predictores: la imputación se realizará dentro de los pipelines para preservar la muestra longitudinal y evitar filtración de información durante la validación cruzada.


In [ ]:
ids = ["CODUSU", "NRO_HOGAR", "COMPONENTE"]

historia_2024 = (
    base_X_2024[ids + ["informal"]]
    .rename(columns={"informal": "informal_2024"})
    .copy()
)

base_2025_modelo = (
    base_X_2025[ids + ["informal"] + features_economicas]
    .rename(columns={"informal": "informal_2025"})
    .copy()
)

assert historia_2024.duplicated(ids).sum() == 0
assert base_2025_modelo.duplicated(ids).sum() == 0

base_tp4 = base_2025_modelo.merge(
    historia_2024,
    on=ids,
    how="inner",
    validate="one_to_one",
)

features_tp4 = features_economicas + ["informal_2024"]
X_tp4 = base_tp4[features_tp4].copy()
y_tp4 = base_tp4["informal_2025"].astype(int).copy()

print("Muestra longitudinal:", len(base_tp4))
print("Shape X:", X_tp4.shape)
print("Shape y:", y_tp4.shape)
print("Predictores conceptuales finales:", len(features_tp4))
print("\nDistribución de y_2025 (%):")
print(y_tp4.value_counts(normalize=True).sort_index().mul(100).round(2))


In [ ]:
control_tp4 = pd.DataFrame({
    "tipo": X_tp4.dtypes.astype(str),
    "n_unicos": X_tp4.nunique(dropna=True),
    "faltantes": X_tp4.isna().sum(),
    "faltantes_pct": (X_tp4.isna().mean() * 100).round(2),
}).sort_values("faltantes_pct", ascending=False)

display(control_tp4)

variables_prohibidas = {
    "PP07H", "informal_2025", "y_2025", "EMPLEO", "SECTOR", "CH08"
}
encontradas = sorted(variables_prohibidas.intersection(X_tp4.columns))
assert not encontradas, f"Posible target leakage directo: {encontradas}"
assert y_tp4.isna().sum() == 0
assert X_tp4["informal_2024"].isna().sum() == 0

print("\nVariables prohibidas encontradas:", encontradas)
print("OK: matriz longitudinal lista para preprocesamiento.")


## 5. Preprocesamiento

- Numéricas: imputación por mediana.
- Binarias: imputación por moda.
- Categóricas: imputación por moda + one-hot encoding con una categoría de referencia.
- Para Logit/LASSO/Ridge, toda la matriz transformada se estandariza antes de estimar.
- Para CART se utiliza la misma imputación y codificación, sin estandarización porque el árbol es invariante a la escala.

En validación cruzada, el preprocesamiento se ejecuta **dentro de cada fold**.


In [ ]:
features_binarias_final = features_binarias + ["informal_2024"]

pipeline_numericas = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

pipeline_binarias = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

pipeline_categoricas = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        drop="first",
        handle_unknown="ignore",
        sparse_output=False,
    )),
])

transformador = ColumnTransformer([
    ("numericas", pipeline_numericas, features_numericas),
    ("binarias", pipeline_binarias, features_binarias_final),
    ("categoricas", pipeline_categoricas, features_categoricas),
])

preprocesador_logit = Pipeline([
    ("transformador", transformador),
    ("scaler", StandardScaler()),
])

X_auditoria = preprocesador_logit.fit_transform(X_tp4)
nombres_transformados = (
    preprocesador_logit
    .named_steps["transformador"]
    .get_feature_names_out()
)

print("Shape original:", X_tp4.shape)
print("Shape post encoding:", X_auditoria.shape)
print("NaN post procesamiento:", np.isnan(X_auditoria).sum())
print("Columnas modelables post encoding:", len(nombres_transformados))


In [ ]:
def limpiar_nombres_features(nombres):
    return (
        pd.Series(nombres)
        .str.replace("numericas__", "", regex=False)
        .str.replace("binarias__", "", regex=False)
        .str.replace("categoricas__", "", regex=False)
    )

nombres_limpios = limpiar_nombres_features(nombres_transformados)

# Categorías de referencia de las variables categóricas
_ohe = (
    preprocesador_logit
    .named_steps["transformador"]
    .named_transformers_["categoricas"]
    .named_steps["onehot"]
)

categorias_referencia = pd.DataFrame({
    "variable": features_categoricas,
    "categoria_referencia": [cats[0] for cats in _ohe.categories_],
})

display(categorias_referencia)


## 6. A. Regresión logística con regularización

### A.1. Trayectorias de coeficientes

Se utiliza la grilla requerida:

$$
\lambda = 10^n,\qquad n\in\{-5,-4,\ldots,4,5\}
$$

En `LogisticRegression`, $C=1/\lambda$. A medida que aumenta $\lambda$, LASSO puede llevar coeficientes exactamente a cero, mientras Ridge los contrae sin selección exacta.


In [ ]:
lambdas = 10.0 ** np.arange(-5, 6)
Cs = 1 / lambdas

# Transformación completa sólo para visualizar las trayectorias.
# La selección de lambda se realizará luego con el preprocesamiento dentro de cada fold.
X_std = preprocesador_logit.fit_transform(X_tp4)
y = y_tp4.to_numpy()
nombres_path = limpiar_nombres_features(
    preprocesador_logit.named_steps["transformador"].get_feature_names_out()
)

coef_lasso_path = []
coef_ridge_path = []

for C in Cs:
    lasso = LogisticRegression(
        penalty="l1", C=C, solver="liblinear",
        max_iter=10000, random_state=RANDOM_STATE,
    ).fit(X_std, y)

    ridge = LogisticRegression(
        penalty="l2", C=C, solver="lbfgs",
        max_iter=10000, random_state=RANDOM_STATE,
    ).fit(X_std, y)

    coef_lasso_path.append(lasso.coef_[0])
    coef_ridge_path.append(ridge.coef_[0])

coef_lasso_path = np.asarray(coef_lasso_path)
coef_ridge_path = np.asarray(coef_ridge_path)

print("Grilla lambda:", lambdas)
print("LASSO path:", coef_lasso_path.shape)
print("Ridge path:", coef_ridge_path.shape)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
x = np.log10(lambdas)

idx_historia = list(nombres_path).index("informal_2024")

for j in range(coef_lasso_path.shape[1]):
    axes[0].plot(x, coef_lasso_path[:, j], linewidth=0.8, alpha=0.45)
axes[0].plot(x, coef_lasso_path[:, idx_historia], linewidth=3, label="informal_2024")
axes[0].axhline(0, linewidth=0.8)
axes[0].set_title("A. LASSO (L1)")
axes[0].set_xlabel("log10(lambda)")
axes[0].set_ylabel("Coeficiente estandarizado")
axes[0].legend()

for j in range(coef_ridge_path.shape[1]):
    axes[1].plot(x, coef_ridge_path[:, j], linewidth=0.8, alpha=0.45)
axes[1].plot(x, coef_ridge_path[:, idx_historia], linewidth=3, label="informal_2024")
axes[1].axhline(0, linewidth=0.8)
axes[1].set_title("B. Ridge (L2)")
axes[1].set_xlabel("log10(lambda)")
axes[1].legend()

fig.suptitle("Trayectorias de coeficientes según la fuerza de penalización")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "A1_trayectorias_lasso_ridge.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
resumen_sparsity = pd.DataFrame({
    "lambda": lambdas,
    "C": Cs,
    "coeficientes_cero": (np.abs(coef_lasso_path) < 1e-8).sum(axis=1),
})
resumen_sparsity["proporcion_cero"] = (
    resumen_sparsity["coeficientes_cero"] / coef_lasso_path.shape[1]
)

display(resumen_sparsity)

plt.figure(figsize=(8, 4.5))
plt.plot(
    np.log10(resumen_sparsity["lambda"]),
    resumen_sparsity["proporcion_cero"],
    marker="o",
)
plt.xlabel("log10(lambda)")
plt.ylabel("Proporción de coeficientes en cero")
plt.title("Selección de variables de LASSO")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "A1_proporcion_ceros_lasso.png", dpi=180, bbox_inches="tight")
plt.show()


### A.2. Penalidad óptima mediante 5-fold cross-validation

Para cada $\lambda$ se calcula el error de clasificación en cinco folds estratificados:

$$
Error = 1 - Accuracy
$$

El preprocesamiento se estima dentro de cada fold.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
resultados_cv = []

for penalidad, penalty, solver in [
    ("LASSO", "l1", "liblinear"),
    ("Ridge", "l2", "lbfgs"),
]:
    for lam, C in zip(lambdas, Cs):
        modelo = Pipeline([
            ("preprocesamiento", clone(preprocesador_logit)),
            ("modelo", LogisticRegression(
                penalty=penalty,
                C=C,
                solver=solver,
                max_iter=10000,
                random_state=RANDOM_STATE,
            )),
        ])

        accuracy_folds = cross_val_score(
            modelo,
            X_tp4,
            y_tp4,
            cv=cv,
            scoring="accuracy",
            n_jobs=-1,
        )

        for fold, accuracy in enumerate(accuracy_folds, start=1):
            resultados_cv.append({
                "penalidad": penalidad,
                "lambda": lam,
                "C": C,
                "fold": fold,
                "accuracy": accuracy,
                "error": 1 - accuracy,
            })

resultados_cv = pd.DataFrame(resultados_cv)


In [ ]:
resumen_cv = (
    resultados_cv
    .groupby(["penalidad", "lambda", "C"], as_index=False)
    .agg(
        error_medio=("error", "mean"),
        error_sd=("error", "std"),
        accuracy_media=("accuracy", "mean"),
    )
)

mejores_lambda = (
    resumen_cv
    .sort_values(["penalidad", "error_medio", "lambda"], ascending=[True, True, False])
    .groupby("penalidad", as_index=False)
    .first()
)

display(mejores_lambda.round(5))
mejores_lambda.to_csv(OUTPUT_DIR / "A2_lambda_cv.csv", index=False)

lambda_lasso_cv = float(mejores_lambda.loc[mejores_lambda["penalidad"] == "LASSO", "lambda"].iloc[0])
lambda_ridge_cv = float(mejores_lambda.loc[mejores_lambda["penalidad"] == "Ridge", "lambda"].iloc[0])
C_lasso_cv = 1 / lambda_lasso_cv
C_ridge_cv = 1 / lambda_ridge_cv

print("LASSO: lambda_cv =", lambda_lasso_cv, "| C =", C_lasso_cv)
print("Ridge: lambda_cv =", lambda_ridge_cv, "| C =", C_ridge_cv)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, penalidad in zip(axes, ["LASSO", "Ridge"]):
    datos = resultados_cv[resultados_cv["penalidad"] == penalidad]
    posiciones = np.arange(len(lambdas))
    valores = [
        datos.loc[datos["lambda"] == lam, "error"].values
        for lam in lambdas
    ]

    ax.boxplot(valores, positions=posiciones)
    ax.set_xticks(posiciones)
    ax.set_xticklabels(
        [f"10^{int(np.log10(lam))}" for lam in lambdas],
        rotation=45,
    )
    ax.set_title(penalidad)
    ax.set_xlabel("lambda")
    ax.set_ylabel("Error de clasificación (1 - Accuracy)")

fig.suptitle("Distribución del error de validación por penalidad")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "A2_boxplots_error_lambda.png", dpi=180, bbox_inches="tight")
plt.show()


### A.3. Estimación con la penalidad óptima y comparación de coeficientes

Se estiman Logit sin penalización, LASSO con $\lambda_{CV}$ y Ridge con $\lambda_{CV}$ sobre la misma matriz estandarizada. La tabla completa se exporta para el apéndice del informe.


In [ ]:
logit_final = Pipeline([
    ("preprocesamiento", clone(preprocesador_logit)),
    ("modelo", LogisticRegression(
        penalty=None, solver="lbfgs", max_iter=10000,
        random_state=RANDOM_STATE,
    )),
])

lasso_final = Pipeline([
    ("preprocesamiento", clone(preprocesador_logit)),
    ("modelo", LogisticRegression(
        penalty="l1", C=C_lasso_cv, solver="liblinear",
        max_iter=10000, random_state=RANDOM_STATE,
    )),
])

ridge_final = Pipeline([
    ("preprocesamiento", clone(preprocesador_logit)),
    ("modelo", LogisticRegression(
        penalty="l2", C=C_ridge_cv, solver="lbfgs",
        max_iter=10000, random_state=RANDOM_STATE,
    )),
])

for modelo in [logit_final, lasso_final, ridge_final]:
    modelo.fit(X_tp4, y_tp4)

transformador_estimado = (
    logit_final.named_steps["preprocesamiento"]
    .named_steps["transformador"]
)
nombres_finales = limpiar_nombres_features(transformador_estimado.get_feature_names_out())

tabla_coeficientes = pd.DataFrame({
    "variable": nombres_finales,
    "Logit": logit_final.named_steps["modelo"].coef_[0],
    "LASSO": lasso_final.named_steps["modelo"].coef_[0],
    "Ridge": ridge_final.named_steps["modelo"].coef_[0],
})

tabla_coeficientes["LASSO_elimina"] = np.isclose(
    tabla_coeficientes["LASSO"], 0, atol=1e-8
)
tabla_coeficientes["abs_LASSO"] = tabla_coeficientes["LASSO"].abs()
tabla_coeficientes = tabla_coeficientes.sort_values("abs_LASSO", ascending=False).reset_index(drop=True)

display(tabla_coeficientes.head(20).round(4))
print("Coeficientes totales:", len(tabla_coeficientes))
print("Eliminados por LASSO:", tabla_coeficientes["LASSO_elimina"].sum())
print("Conservados por LASSO:", (~tabla_coeficientes["LASSO_elimina"]).sum())

tabla_coeficientes.to_csv(OUTPUT_DIR / "A3_coeficientes_logit_lasso_ridge.csv", index=False)


In [ ]:
print("Variables eliminadas exactamente por LASSO:")
display(
    tabla_coeficientes.loc[
        tabla_coeficientes["LASSO_elimina"],
        ["variable", "Logit", "LASSO", "Ridge"],
    ].round(4)
)

print("\nCoeficiente de historia laboral:")
display(
    tabla_coeficientes.loc[
        tabla_coeficientes["variable"] == "informal_2024",
        ["variable", "Logit", "LASSO", "Ridge"],
    ].round(4)
)


## 7. B. Árbol de clasificación CART

### B.1. Selección de complejidad y poda

Se evalúa `ccp_alpha` mediante 5-fold cross-validation. Valores mayores implican mayor poda. La grilla llega hasta 1 para observar también la zona de sobrepoda y evitar que el óptimo quede artificialmente en el borde de búsqueda.


In [ ]:
# Para CART se usa el transformador sin StandardScaler.
preprocesador_cart = clone(transformador)
ccp_grid = np.concatenate([[0.0], np.logspace(-5, 0, 45)])
resultados_cart_cv = []

for alpha in ccp_grid:
    cart = Pipeline([
        ("preprocesamiento", clone(preprocesador_cart)),
        ("modelo", DecisionTreeClassifier(
            ccp_alpha=alpha,
            random_state=RANDOM_STATE,
        )),
    ])

    for fold, (idx_train, idx_val) in enumerate(cv.split(X_tp4, y_tp4), start=1):
        X_train = X_tp4.iloc[idx_train]
        X_val = X_tp4.iloc[idx_val]
        y_train = y_tp4.iloc[idx_train]
        y_val = y_tp4.iloc[idx_val]

        cart.fit(X_train, y_train)
        pred = cart.predict(X_val)
        arbol = cart.named_steps["modelo"]

        resultados_cart_cv.append({
            "ccp_alpha": alpha,
            "fold": fold,
            "accuracy": accuracy_score(y_val, pred),
            "error": 1 - accuracy_score(y_val, pred),
            "n_hojas": arbol.get_n_leaves(),
            "profundidad": arbol.get_depth(),
        })

resultados_cart_cv = pd.DataFrame(resultados_cart_cv)

resumen_cart = (
    resultados_cart_cv
    .groupby("ccp_alpha", as_index=False)
    .agg(
        error_medio=("error", "mean"),
        error_sd=("error", "std"),
        accuracy_media=("accuracy", "mean"),
        hojas_promedio=("n_hojas", "mean"),
        profundidad_promedio=("profundidad", "mean"),
    )
)

min_error = resumen_cart["error_medio"].min()
candidatos = resumen_cart[np.isclose(resumen_cart["error_medio"], min_error, atol=1e-12)]
alpha_optimo = float(candidatos["ccp_alpha"].max())

fila_optima = resumen_cart.loc[resumen_cart["ccp_alpha"] == alpha_optimo]
display(fila_optima.round(6))


In [ ]:
grafico_cart = resumen_cart[resumen_cart["ccp_alpha"] > 0].copy()

plt.figure(figsize=(9, 5))
plt.plot(grafico_cart["ccp_alpha"], grafico_cart["error_medio"], marker="o")
plt.axvline(alpha_optimo, linestyle="--", label=f"ccp_alpha CV = {alpha_optimo:.5g}")
plt.xscale("log")
plt.xlabel("ccp_alpha")
plt.ylabel("Error de clasificación (1 - Accuracy)")
plt.title("Selección de complejidad de CART mediante 5-fold CV")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "B1_cv_ccp_alpha.png", dpi=180, bbox_inches="tight")
plt.show()


### B.2. Árbol podado e importancia de predictores

El árbol final se estima con el `ccp_alpha` seleccionado. Se reporta la importancia por columna transformada y también una agregación por predictor conceptual para facilitar la interpretación económica.


In [ ]:
cart_final = Pipeline([
    ("preprocesamiento", clone(preprocesador_cart)),
    ("modelo", DecisionTreeClassifier(
        ccp_alpha=alpha_optimo,
        random_state=RANDOM_STATE,
    )),
])
cart_final.fit(X_tp4, y_tp4)

arbol_final = cart_final.named_steps["modelo"]
transformador_cart_estimado = cart_final.named_steps["preprocesamiento"]
nombres_cart = limpiar_nombres_features(transformador_cart_estimado.get_feature_names_out())

print("ccp_alpha:", alpha_optimo)
print("Profundidad:", arbol_final.get_depth())
print("Hojas:", arbol_final.get_n_leaves())
print("Nodos:", arbol_final.tree_.node_count)


In [ ]:
importancias_cart = pd.DataFrame({
    "variable": nombres_cart,
    "importancia": arbol_final.feature_importances_,
}).sort_values("importancia", ascending=False).reset_index(drop=True)

importancias_positivas = importancias_cart[importancias_cart["importancia"] > 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={"width_ratios": [1.6, 1]})

plot_tree(
    arbol_final,
    feature_names=list(nombres_cart),
    class_names=["Formal", "Informal"],
    filled=True,
    rounded=True,
    proportion=True,
    fontsize=8,
    ax=axes[0],
)
axes[0].set_title("A. Árbol podado")

mostrar = importancias_positivas.head(20).iloc[::-1]
axes[1].barh(mostrar["variable"], mostrar["importancia"])
axes[1].set_xlabel("Importancia")
axes[1].set_title("B. Principales importancias")

fig.suptitle("CART para informalidad laboral")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "B2_arbol_importancias.png", dpi=180, bbox_inches="tight")
plt.show()

display(importancias_positivas.head(25).round(4))


In [ ]:
def predictor_original(nombre):
    directas = features_numericas + features_binarias_final
    if nombre in directas:
        return nombre
    for var in features_categoricas:
        if nombre.startswith(var + "_"):
            return var
    return nombre

importancias_cart["predictor_original"] = importancias_cart["variable"].apply(predictor_original)
importancia_conceptual = (
    importancias_cart
    .groupby("predictor_original", as_index=False)["importancia"]
    .sum()
    .sort_values("importancia", ascending=False)
)

display(importancia_conceptual.round(4))
importancia_conceptual.to_csv(OUTPUT_DIR / "B2_importancia_conceptual_cart.csv", index=False)


In [ ]:
comparacion_lasso_cart = (
    tabla_coeficientes[["variable", "LASSO_elimina"]]
    .merge(importancias_cart[["variable", "importancia"]], on="variable", how="left")
)
comparacion_lasso_cart["CART_importancia_cero"] = comparacion_lasso_cart["importancia"].fillna(0).eq(0)

resumen_seleccion = pd.crosstab(
    comparacion_lasso_cart["LASSO_elimina"],
    comparacion_lasso_cart["CART_importancia_cero"],
)

print("Cruce de selección LASSO vs. uso en CART:")
display(resumen_seleccion)


## 8. C. Comparación entre métodos

### C.1. Desempeño predictivo

Los cuatro modelos se comparan con predicciones out-of-fold de los mismos 5 folds y umbral $p\geq0,5$. Se reportan matriz de confusión, Accuracy, $1-Accuracy$, Precision, Recall, F1 y AUC.

**Nota metodológica:** los hiperparámetros de regularización y poda fueron seleccionados mediante CV sobre esta muestra. Una evaluación completamente independiente requeriría una tercera muestra o nested cross-validation; esta limitación se reporta al final.


In [ ]:
modelos = {
    "Logit": Pipeline([
        ("preprocesamiento", clone(preprocesador_logit)),
        ("modelo", LogisticRegression(
            penalty=None, solver="lbfgs", max_iter=10000,
            random_state=RANDOM_STATE,
        )),
    ]),
    "LASSO": Pipeline([
        ("preprocesamiento", clone(preprocesador_logit)),
        ("modelo", LogisticRegression(
            penalty="l1", C=C_lasso_cv, solver="liblinear",
            max_iter=10000, random_state=RANDOM_STATE,
        )),
    ]),
    "Ridge": Pipeline([
        ("preprocesamiento", clone(preprocesador_logit)),
        ("modelo", LogisticRegression(
            penalty="l2", C=C_ridge_cv, solver="lbfgs",
            max_iter=10000, random_state=RANDOM_STATE,
        )),
    ]),
    "CART": Pipeline([
        ("preprocesamiento", clone(preprocesador_cart)),
        ("modelo", DecisionTreeClassifier(
            ccp_alpha=alpha_optimo,
            random_state=RANDOM_STATE,
        )),
    ]),
}

resultados_oof = {}
filas_metricas = []

for nombre, modelo in modelos.items():
    prob = cross_val_predict(
        modelo,
        X_tp4,
        y_tp4,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]
    pred = (prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_tp4, pred).ravel()
    resultados_oof[nombre] = {"prob": prob, "pred": pred}

    filas_metricas.append({
        "Modelo": nombre,
        "VN": tn,
        "FP": fp,
        "FN": fn,
        "VP": tp,
        "Accuracy": accuracy_score(y_tp4, pred),
        "1-Accuracy": 1 - accuracy_score(y_tp4, pred),
        "Precision": precision_score(y_tp4, pred),
        "Recall": recall_score(y_tp4, pred),
        "F1": f1_score(y_tp4, pred),
        "AUC": roc_auc_score(y_tp4, prob),
    })

tabla_metricas = pd.DataFrame(filas_metricas)
display(tabla_metricas.round(4))
tabla_metricas.to_csv(OUTPUT_DIR / "C1_metricas_modelos.csv", index=False)


In [ ]:
plt.figure(figsize=(8, 6))

for nombre in modelos:
    prob = resultados_oof[nombre]["prob"]
    fpr, tpr, _ = roc_curve(y_tp4, prob)
    auc = roc_auc_score(y_tp4, prob)
    plt.plot(fpr, tpr, linewidth=2, label=f"{nombre} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
plt.xlabel("Tasa de falsos positivos")
plt.ylabel("Tasa de verdaderos positivos (Recall)")
plt.title("Curvas ROC — comparación de modelos")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "C1_curvas_roc.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for ax, nombre in zip(axes.ravel(), modelos):
    ConfusionMatrixDisplay.from_predictions(
        y_tp4,
        resultados_oof[nombre]["pred"],
        display_labels=["Formal", "Informal"],
        ax=ax,
        colorbar=False,
    )
    ax.set_title(nombre)

fig.suptitle("Matrices de confusión — umbral p >= 0,5")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "C1_matrices_confusion.png", dpi=180, bbox_inches="tight")
plt.show()


### C.1.b. Identificación de informales en una región argentina

Se evalúan las mismas predicciones out-of-fold en una región específica. Por defecto se utiliza **GBA**; puede cambiarse la variable `REGION_EVALUAR` si se decide presentar otra región.


In [ ]:
REGION_EVALUAR = "gba"
mask_region = X_tp4["region"].eq(REGION_EVALUAR).to_numpy()
y_region = y_tp4.to_numpy()[mask_region]

print("Región evaluada:", REGION_EVALUAR)
print("Observaciones:", mask_region.sum())
print("Informales:", y_region.sum())
print("Tasa de informalidad (%):", round(y_region.mean() * 100, 2))

filas_region = []
for nombre in modelos:
    prob = resultados_oof[nombre]["prob"][mask_region]
    pred = resultados_oof[nombre]["pred"][mask_region]
    tn, fp, fn, tp = confusion_matrix(y_region, pred).ravel()

    filas_region.append({
        "Modelo": nombre,
        "VN": tn,
        "FP": fp,
        "FN": fn,
        "VP": tp,
        "Accuracy": accuracy_score(y_region, pred),
        "1-Accuracy": 1 - accuracy_score(y_region, pred),
        "Precision": precision_score(y_region, pred, zero_division=0),
        "Recall": recall_score(y_region, pred, zero_division=0),
        "F1": f1_score(y_region, pred, zero_division=0),
        "AUC": roc_auc_score(y_region, prob) if len(np.unique(y_region)) == 2 else np.nan,
    })

tabla_region = pd.DataFrame(filas_region)
display(tabla_region.round(4))
tabla_region.to_csv(OUTPUT_DIR / f"C1_metricas_region_{REGION_EVALUAR}.csv", index=False)


### C.2. Selección del modelo para política pública

Para la Secretaría de Trabajo, un **falso negativo** es un trabajador informal que el modelo no identifica. Si el costo social de omitir a un informal es mayor que el costo de incluir erróneamente a un formal, el Recall y la cantidad de falsos negativos deben recibir especial atención. La elección final debe balancear:

- identificación de informales (Recall / FN),
- capacidad discriminatoria global (AUC),
- error total ($1-Accuracy$),
- parsimonia e interpretabilidad.


In [ ]:
ranking_politica = tabla_metricas[
    ["Modelo", "FN", "FP", "Recall", "F1", "AUC", "1-Accuracy"]
].sort_values(["FN", "AUC"], ascending=[True, False])

display(ranking_politica.round(4))

mejor_recall = tabla_metricas.sort_values("Recall", ascending=False).iloc[0]
print(
    "Mayor Recall:", mejor_recall["Modelo"],
    "| Recall =", round(mejor_recall["Recall"], 4),
    "| FN =", int(mejor_recall["FN"]),
)


## 9. Síntesis para redactar el informe

Completar luego de ejecutar toda la notebook:

1. **Regularización:** ¿cómo cambian los coeficientes al aumentar $\lambda$? ¿Cuántos lleva LASSO a cero con $\lambda_{CV}$? ¿Ridge elimina alguno?
2. **Persistencia:** ¿qué lugar ocupa `informal_2024` en Logit/LASSO/Ridge y en CART?
3. **Estructura económica:** ¿qué variables laborales/productivas aparecen entre las de mayor magnitud o importancia?
4. **CART:** ¿la mayor flexibilidad no lineal mejora la capacidad predictiva o principalmente cambia el trade-off Precision/Recall?
5. **Política pública:** ¿qué modelo minimiza falsos negativos y cuál ofrece el mejor AUC?
6. **Región:** ¿cambia la identificación de informales en GBA respecto del total nacional?

**Interpretación:** los resultados son predictivos/asociativos, no causales.


## 10. D. Herramientas de IA y reflexión final

### Reflexión — máximo un párrafo

> La IA se utilizó como apoyo para ordenar la secuencia del trabajo, revisar la construcción de variables de la EPH, detectar posibles problemas de target leakage, depurar código y discutir la interpretación de regularización, CART y métricas de clasificación. Su aporte fue especialmente útil para estructurar el flujo de programación y contrastar alternativas metodológicas. La lectura de códigos específicos de la EPH y las decisiones finales de especificación se verificaron contra la documentación y las salidas efectivamente obtenidas. La IA no sustituyó la ejecución ni la validación de los resultados.

### Apéndice de prompts
Copiar textualmente, en orden cronológico, todos los prompts utilizados para resolver el TP4 e indicar la LLM empleada.

### Tiempo total aproximado
**[COMPLETAR] horas**.


## 11. Checklist de entrega

- [ ] Código ordenado por inciso A.1, A.2, A.3, B.1, B.2, C.1, C.2 y D.
- [ ] Informe final en PDF de **máximo 5 páginas**, sin contar apéndices.
- [ ] Link al repositorio de GitHub en la primera página.
- [ ] Figuras y tablas principales exportadas desde `/content/TP4_outputs/`.
- [ ] Apéndice de coeficientes completos.
- [ ] Apéndice con todos los prompts de IA y nombre de la LLM.
- [ ] Tiempo total aproximado reportado.
- [ ] Commit y push final con el mensaje **“Entrega final del TP”**.
- [ ] No realizar cambios posteriores al commit final.
